## Gemini Pipeline - All Questions
Process all 42 questions from the question bank and store responses in JSONL format.
This notebook queries the Gemini 2.5 Flash model for each question and saves responses to `data/responses/gemini.jsonl`.


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import google.genai as genai

cwd = Path.cwd()
dotenv_path = None
for candidate in [cwd / ".env.local", cwd.parent / ".env.local", cwd.parent.parent / ".env.local"]:
    if candidate.exists():
        dotenv_path = candidate
        break
if not dotenv_path:
    raise ValueError(".env.local not found. Place it in the repo root or export GOOGLE_API_KEY manually.")
load_dotenv(dotenv_path)

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Set it in .env.local or export it.")
client = genai.Client(api_key=api_key)

print("Loaded Gemini client")


Loaded Gemini client


In [2]:
import json
import time
from datetime import datetime, timezone
import pandas as pd

# Load the question bank
question_csv = Path("../figures/question_bank.csv")
questions_df = pd.read_csv(question_csv)

print(f"Loaded {len(questions_df)} questions from {question_csv}")
print(f"Questions: {questions_df['question_id'].tolist()[:5]}... (showing first 5)")

# Define the system prompt
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

# Create output directory
output_dir = Path("../data/responses")
output_dir.mkdir(parents=True, exist_ok=True)

# Process all questions
output_file = output_dir / "gemini.jsonl"
results = {"ok": 0, "error": 0}

with open(output_file, "w", encoding="utf-8") as f:
    for idx, row in questions_df.iterrows():
        question_id = row["question_id"]
        question_text = row["question"]
        figure = row.get("figure", "Unknown")
        
        print(f"[{idx+1}/{len(questions_df)}] Querying {question_id}...", end=" ", flush=True)
        
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=question_text,
                config=genai.types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=1024,
                    system_instruction=SYSTEM_PROMPT,
                ),
            )
            
            response_text = response.text
            tokens = response.usage_metadata.total_token_count
            model_version = "gemini-2.5-flash"
            
            # Create record
            record = {
                "question_id": question_id,
                "figure": figure,
                "model": "gemini",
                "model_version": model_version,
                "prompt": question_text,
                "response": response_text,
                "temperature": 0.0,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "tokens_used": tokens,
            }
            
            f.write(json.dumps(record) + "\n")
            results["ok"] += 1
            print(f"✓ ({tokens} tokens)")
            
        except Exception as e:
            results["error"] += 1
            print(f"✗ ERROR: {e}")
        
        # Rate limiting
        time.sleep(1.0)

print(f"\n=== Complete ===")
print(f"Saved {results['ok']} responses to {output_file}")
if results["error"] > 0:
    print(f"Errors: {results['error']}")


Loaded 42 questions from ../figures/question_bank.csv
Questions: ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']... (showing first 5)
[1/42] Querying Q1... ✓ (1093 tokens)
[2/42] Querying Q2... ✓ (1118 tokens)
[3/42] Querying Q3... ✓ (1114 tokens)
[4/42] Querying Q4... ✓ (1103 tokens)
[5/42] Querying Q5... ✓ (1099 tokens)
[6/42] Querying Q6... ✓ (1128 tokens)
[7/42] Querying Q7... ✗ ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 33.798260575s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas